# Experiment 2 — Encoding x Algorithm Sweep (with class imbalance handling)
Telco Customer Churn

Builds on exp1's baseline. Sweeps **Label Encoding vs One-Hot Encoding** across **4 algorithms**, all configured to handle the churn class imbalance, and logs every combo as a nested MLflow run so they're comparable side-by-side on DagsHub.

In [1]:
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np

import mlflow
import mlflow.sklearn
import dagshub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# ========================== CONFIGURATION ==========================
CONFIG = {
    "data_path": "data.csv",   # local file, same folder as notebook
    "test_size": 0.2,
    "mlflow_tracking_uri": "https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow",
    "dagshub_repo_owner": "rehansarfraz8903",
    "dagshub_repo_name": "Customer-Churn-prediction",
    "experiment_name": "Encoding vs Algorithm Sweep"
}

mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
mlflow.set_experiment(CONFIG["experiment_name"])

Accessing as rehansarfraz8903

Initialized MLflow to track repo "rehansarfraz8903/Customer-Churn-prediction"

Repository rehansarfraz8903/Customer-Churn-prediction initialized!

2026/08/13 00:23:22 INFO mlflow.tracking.fluent: Experiment with name 'Encoding vs Algorithm Sweep' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e0aa666e58e845a6b74ef9446e6d11e3', creation_time=1786605803874, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786605803874, lifecycle_stage='active', name='Encoding vs Algorithm Sweep', tags={}, trace_location=None, workspace='default'>

## Load + base cleaning
(shared by both encodings — only the categorical encoding step differs)

In [3]:
def load_and_clean(path):
    df = pd.read_csv(path)
    df.drop(columns=['customerID'], inplace=True, errors='ignore')

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).infer_objects(copy=False)
    return df

df = load_and_clean(CONFIG["data_path"])
churn_rate = df['Churn'].mean()
print(f"Churn rate: {churn_rate:.2%}")
df.head()

Churn rate: 26.54%


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Feature engineering — Label Encoding vs One-Hot Encoding

Label encoding is what exp1 used (fast, but implies false ordering on nominal columns like `PaymentMethod`). One-hot removes that false ordering at the cost of more columns. We build both `X` matrices here and sweep over them.

In [4]:
def build_label_encoded(df):
    df_enc = df.copy()
    cat_cols = df_enc.select_dtypes(include='object').columns
    for col in cat_cols:
        df_enc[col] = LabelEncoder().fit_transform(df_enc[col])
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
    df_enc[num_cols] = StandardScaler().fit_transform(df_enc[num_cols])
    X = df_enc.drop(columns=['Churn'])
    y = df_enc['Churn']
    return X, y

def build_onehot_encoded(df):
    y = df['Churn']
    X_raw = df.drop(columns=['Churn'])
    cat_cols = X_raw.select_dtypes(include='object').columns.tolist()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    preprocessor = ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ])
    X = preprocessor.fit_transform(X_raw)
    return X, y

ENCODINGS = {
    "LabelEncoding": build_label_encoded(df),
    "OneHotEncoding": build_onehot_encoded(df),
}

## Algorithms — all set up to handle the churn imbalance

- `class_weight='balanced'` for Logistic Regression and Random Forest
- `scale_pos_weight` for XGBoost (ratio of negative/positive class)
- Gradient Boosting included as-is for comparison (no native imbalance handling — worth seeing if it still competes)

In [5]:
neg, pos = (df["Churn"] == 0).sum(), (df["Churn"] == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")

ALGORITHMS = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight,
                              use_label_encoder=False, eval_metric="logloss", random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
}

scale_pos_weight for XGBoost: 2.77


## Train + evaluate every (encoding x algorithm) combo

Logged as nested MLflow runs under one parent run, so you can compare all 8 combos together on DagsHub.

In [6]:
def log_model_params(algo_name, model):
    """Log the hyperparameters relevant to comparing runs."""
    params_to_log = {}
    if algo_name == "LogisticRegression":
        params_to_log["C"] = model.C
        params_to_log["class_weight"] = str(model.class_weight)
    elif algo_name == "RandomForest":
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["class_weight"] = str(model.class_weight)
    elif algo_name == "XGBoost":
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["scale_pos_weight"] = model.scale_pos_weight
    elif algo_name == "GradientBoosting":
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate

    mlflow.log_params(params_to_log)


def train_and_evaluate(encodings, algorithms):
    results = []
    with mlflow.start_run(run_name="All Experiments"):
        for enc_name, (X, y) in encodings.items():
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=CONFIG["test_size"], random_state=42, stratify=y
            )

            for algo_name, algorithm in algorithms.items():
                with mlflow.start_run(run_name=f"{algo_name} with {enc_name}", nested=True):
                    try:
                        mlflow.log_params({
                            "encoding": enc_name,
                            "algorithm": algo_name,
                            "test_size": CONFIG["test_size"],
                        })

                        model = algorithm
                        model.fit(X_train, y_train)
                        log_model_params(algo_name, model)

                        y_pred = model.predict(X_test)
                        metrics = {
                            "accuracy": accuracy_score(y_test, y_pred),
                            "precision": precision_score(y_test, y_pred),
                            "recall": recall_score(y_test, y_pred),
                            "f1_score": f1_score(y_test, y_pred),
                        }
                        mlflow.log_metrics(metrics)
                        mlflow.sklearn.log_model(model, "model")

                        print(f"\n{algo_name} + {enc_name}: {metrics}")
                        results.append({"encoding": enc_name, "algorithm": algo_name, **metrics})

                    except Exception as e:
                        print(f"Error in {algo_name} with {enc_name}: {e}")
                        mlflow.log_param("error", str(e))

    return pd.DataFrame(results)

results_df = train_and_evaluate(ENCODINGS, ALGORITHMS)
results_df.sort_values("recall", ascending=False)

2026/08/13 00:23:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



LogisticRegression + LabelEncoding: {'accuracy': 0.7381121362668559, 'precision': 0.5042444821731749, 'recall': 0.7941176470588235, 'f1_score': 0.616822429906542}
🏃 View run LogisticRegression with LabelEncoding at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1/runs/25750eec700249208d66b04e96803953
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1


2026/08/13 00:24:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



RandomForest + LabelEncoding: {'accuracy': 0.7877927608232789, 'precision': 0.631578947368421, 'recall': 0.48128342245989303, 'f1_score': 0.5462822458270106}
🏃 View run RandomForest with LabelEncoding at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1/runs/6b15fa824e5b48a492a3e36561122ddb
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1


2026/08/13 00:26:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/08/13 00:29:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Error in XGBoost with OneHotEncoding: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier'].
🏃 View run XGBoost with OneHotEncoding at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1/runs/a0c5348d462a4b7296c0da7d51da3c6c
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1


2026/08/13 00:29:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



GradientBoosting + OneHotEncoding: {'accuracy': 0.7955997161107168, 'precision': 0.6442953020134228, 'recall': 0.5133689839572193, 'f1_score': 0.5714285714285714}
🏃 View run GradientBoosting with OneHotEncoding at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1/runs/5ca21bb73c7a4850957157cc8052f198
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1
🏃 View run All Experiments at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1/runs/b22eab92e50e48598946dbc8ff336de3
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/1


,encoding,algorithm,accuracy,precision,recall,f1_score
0,LabelEncoding,LogisticRegression,0.738112,0.504244,0.794118,0.616822
3,OneHotEncoding,LogisticRegression,0.733144,0.498299,0.783422,0.609148
2,LabelEncoding,GradientBoosting,0.806246,0.670034,0.532086,0.593145
5,OneHotEncoding,GradientBoosting,0.795600,0.644295,0.513369,0.571429
1,LabelEncoding,RandomForest,0.787793,0.631579,0.481283,0.546282
4,OneHotEncoding,RandomForest,0.784954,0.626335,0.470588,0.537405


## Quick read

Sort by `recall` first (catching actual churners is the priority here), then check `precision` didn't collapse too far — that tradeoff is what you're really choosing between across these 8 runs.